In [8]:
## Question-2 RAG
############## Part-1 Data Preparation (Load the data, understand the data, clean columns, understand the column data, cleamn)
## dataframe - raw unprocessed data
## extractedData = cleaned data

import os
import pandas as pd
import numpy as np
import google.generativeai as genai
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import userdata
import time
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

#Loading and Init
# API KEY Load the API
API_KEY = userdata.get('API_KEY')
genai.configure(api_key=API_KEY)
print("API loded")

# Load the model
model = genai.GenerativeModel("gemini-2.5-flash-lite")   #   gemini-1.5-flash, 2.5-pro
dataframe = pd.read_csv("data_scientist_salaries.csv")

# we are storing the imp columns required for our data preparation part. impColumns are necessary to answer the questions asked to the RAG.
# our data set has too much data so we need to filter out important data
impCols = [ "Hobby", "OpenSource", "Country", "Student", "Employment", "FormalEducation", "UndergradMajor", "CompanySize", "DevType",
                  "YearsCoding", "Salary", "SalaryType", "ConvertedSalary"]
embeddingModel = SentenceTransformer('all-MiniLM-L6-v2')


#Understand the data
# clean this dataset and handle null
def preprocessing(dataframe, impCols):

  # extract the data from the dataset only of impCols initialised above
  extractedData = dataframe[impCols].copy()

  #### Handling null or empty values
  for col in impCols:

    # identify if the extracted data is null or not and count of missing values > 0
    if extractedData[col].isnull().sum() > 0:
      # Missing values are present
      # print(col)

      # for missing data in text type colums datatype = object we replace it with "unknown"
      if extractedData[col].dtype == 'object':
        # if object is null add replace the cell with "unknown"
        extractedData[col] = extractedData[col].fillna('unknown')
        # converting it into lower case - ONLY for text columns
        extractedData[col] = extractedData[col].astype(str).str.lower()
        # extractedData[col] = str(extractedData[col]).lower()

      # for missing data in number type columns
      elif col == "ConvertedSalary":
        # if any of the cells from ConvertedSalary column are missing then add the median as the default value
        extractedData[col] = extractedData[col].fillna(extractedData[col].median())

      # For any other columns empty values will be replaced by 0
      else:
        extractedData[col] = extractedData[col].fillna(0)

  if 'ConvertedSalary' in extractedData.columns:
    # converting values to number "50" -> 50 and if there is a value that cant be converted to a number then put NaN.
    extractedData['ConvertedSalary'] = pd.to_numeric(extractedData['ConvertedSalary'], errors='coerce')
    extractedData['ConvertedSalary'] = extractedData['ConvertedSalary'].fillna(extractedData['ConvertedSalary'].median())

  #### Generalising the years coding data to handle all the data range values
  if 'YearsCoding' in extractedData.columns:

    # group years of experience. this is useful for any math related operations or even answering que related to ranges like below 5years
    # midpoint values will be used. bcz this will give us equal spacing between the levels
    yearsexp = {
        "0-2 years":1, "3-5 years": 4, "6-8 years": 7, "9-11 years": 10, "12-14 years": 13, "15-17 years": 16, "18-20 years": 19,
        "21-23 years":22, "24-26 years": 25, "27-29 years": 28, "30 or more years":30
    }
    # take YearsCoding column, convert it to corresponding num value and then map it to the corresp num value in the yearsexp dict
    extractedData['YearsCodingNum'] = extractedData['YearsCoding'].map(yearsexp)

    # if there are any values like null replace those with median value of YearsCodingNum like if the other vals were 1, 4, 7 then fill null values with median (1,4,7)
    extractedData['YearsCodingNum'] = extractedData['YearsCodingNum'].fillna(extractedData['YearsCodingNum'].median())

  # this extractedData is our cleaned version of original dataframe
  return extractedData

# Compute Embeddings
# We are computing embeddings for all the cells and rows present in the csv file. This is done so that after a question is asked by the user we can retrieve the
# answer quickly. A text representation can be done which can be used later for semantic search and retrieval of answers ()

def getEmbeddingsofCSV(extractedData):
  # 1. Convert csv to text
  completeText = []
  for _, rowData in extractedData.iterrows():
    # we are converting table data to text so that we can search on this text later.
    retrievedText = []
    if 'ConvertedSalary' in rowData:
      if rowData['ConvertedSalary'] > 0:
        retrievedText.append(f"Converted Salary -> "+str(rowData['ConvertedSalary']))
    if 'YearsCoding' in rowData:
      # if rowData['YearsCoding'] is not None and str(rowData['YearsCoding']).strip():
      if str(rowData['YearsCoding']).strip():
        retrievedText.append(f"Years Coding -> "+str(rowData['YearsCoding']))
        # print("years coding retrieved rows = "+str(len(rowData['YearsCoding'])))
        if 'YearsCodingNum' in rowData:
          retrievedText.append(f"Years Coding Num -> "+str(rowData['YearsCodingNum']))
    if 'Country' in rowData:
      if rowData['Country'] is not None:
        retrievedText.append(f"Country -> "+str(rowData['Country']))
    if 'CompanySize' in rowData:
      # if rowData['CompanySize'] is not None and str(rowData['CompanySize']).strip():
      if str(rowData['CompanySize']).strip():
        retrievedText.append(f"Company Size -> "+str(rowData['CompanySize']))
    if 'DevType' in rowData:
      if rowData['DevType'] is not None:
        retrievedText.append(f"DevType -> "+str(rowData['DevType']))
    if 'FormalEducation' in rowData:
      if rowData['FormalEducation'] is not None:
        retrievedText.append(f"FormalEducation -> "+str(rowData['FormalEducation']))

    completeText.append("; ".join(retrievedText))

  # Compute the embeddings for semantic search for the complete Text we got from csv
  textembeddingsFromCSV = embeddingModel.encode(completeText)
  return completeText, textembeddingsFromCSV

#####Matching functions or Retriever Functions

## 1. Keyword Matching
def keywordMatching(userQuery, textRep, extractedData):

  eng_stopwords = set(stopwords.words('english'))
  queryWords = []

  # first we extract every unique word from the userQuery and only words which are not stopwords
  for word in userQuery.lower().split():
    clean_word = word.strip(",.!?:;()[]{}")
    # check if word is not a stopword AND has meaningful length (>2 characters)
    if clean_word not in eng_stopwords and len(clean_word) > 2:
      queryWords.append(clean_word)

  # FALLBACK: If all words were stopwords, use original words (but still filter very short ones)
  if not queryWords:
    # Create fallback list with words longer than 2 characters
    fallback_words = []
    for word in userQuery.lower().split():
      clean_word = word.strip(",.!?:;()[]{}")
      if len(clean_word) > 2:
        fallback_words.append(clean_word)
    queryWords = fallback_words

  keywordMatchScores = []

  # compare words in textRep = ["Role: Data Scientist; Experience: 5-10 years; ...] and in the query get a score
  for index, rowData in enumerate(textRep):
    scoreObtained = 0      # number of matching terms or words found in a row
    for word in queryWords:
        # word should be in a row of CSV then only increase score of that row
        # Use word in rowData.lower() for matching
        if word in rowData.lower():
          scoreObtained += 1

    # Only add rows that have at least one match (score > 0)
    if scoreObtained > 0:
      keywordMatchScores.append((scoreObtained, index))   # score= no of matching words and index = row number

  # sort with highest matching row first
  keywordMatchScores.sort(reverse = True)

  temp = []
  for kmscore, index in keywordMatchScores:
    if kmscore > 0:
      temp.append(index)
  indexesFinal = temp

  # get indexes of rows with score match above 0
  if indexesFinal:
    return extractedData.iloc[indexesFinal].copy()    # this is returning the actual data from the CSV that is present in those indexes
  else:
    return extractedData.head(0)

## 2.
def semanticMatching(userQuery, EmbeddingsofCSV, extractedData):
  # convert the text into vector embedding. Embedding contains the semantic meaning and will help in matching
  embeddingofQuery = embeddingModel.encode([userQuery])
  similarityScoreValues = cosine_similarity(embeddingofQuery, EmbeddingsofCSV)[0]   # cosine similaroty would compare both and compare how similar both the vectors are. Same = 1, diff=-1
  indexesFinal = np.argsort(similarityScoreValues)[::-1][:5]    # returning indicies of highest similarity scores. only 50rows
  return extractedData.iloc[indexesFinal].copy()


# ans Questions
def retriever(userQuestion, method, textRep, extractedData, EmbeddingsofCSV):

  # Implementing the question answering part to test the RAG
  # print("Query = "+str(userQuestion))
  # method could be semantic, keyword matching or hybrid
  # print("Method of ans retrieval = "+str(method))

  if method.lower() == 'keyword':
    dataObtained = keywordMatching(userQuestion, textRep, extractedData)

  elif method.lower() == 'semantic':
    dataObtained = semanticMatching(userQuestion, EmbeddingsofCSV, extractedData)

  print("len of dataObtained retrieved data count = "+str(len(dataObtained)))
  contextList = []
  resFinal = []
  # take the dataObtained using those search methods and format that data to be able to use it as context.
  # loop through each row in dataObtained
  for i, (_, rowVal) in enumerate(dataObtained.iterrows()):

    contextList.append(f"\n {i+1}")

    # getting the context
    if rowVal['DevType'] != 'unknown':
        contextList.append(f" Persons Role: {rowVal['DevType']}")

    if rowVal['YearsCoding'] != 'unknown':
        contextList.append(f"  YearsCoding: {rowVal['YearsCoding']}")

    if rowVal['Country'] != 'unknown':
        contextList.append(f"  Country: {rowVal['Country']}")

    if rowVal['ConvertedSalary'] > 0:
        contextList.append(f"  Salary: ${rowVal['ConvertedSalary']:,.0f}")

    # if rowVal['CompanySize']:
    if rowVal['CompanySize'] != 'unknown':
        contextList.append(f"  Company Size: {rowVal['CompanySize']}")

    # if rowVal['FormalEducation'] :
    if rowVal['FormalEducation'] != 'unknown':
        contextList.append(f"  Education: {rowVal['FormalEducation']}")

    # if rowVal['Employment']:
    if rowVal['Employment'] != 'unknown':
      contextList.append(f"  Employment: {rowVal['Employment']}")

  # get the full context as a string
  context = "\n".join(contextList)

  #### Creating a prompt that is clear and understandable
  prompt = "Hello. You are a data science analyst. There is some csv data given to you."
  prompt += "You have to analyse the data and correctly answer the questions asked 'only' from the data given to you."
  prompt += "For salary related questions analyse the data calculate averages and ranges"
  prompt += context
  prompt += "The question is"
  prompt += userQuestion


  # get the response for the given prompt
  try:
    response = model.generate_content(prompt)
    response_text = response.text
  except Exception as e:
    if "429" in str(e):
      response_text = "API quota exceeded."
    else:
      response_text = f"Error: {str(e)}"

  res = {
      'question': userQuestion,
      'dataretrievalMethod': method,
      'dataObtained': dataObtained,
      'response': response_text
  }
  resFinal.append(res)
  return resFinal

# Test the system
# print("Start program")
extractedData = preprocessing(dataframe, impCols)
textRep, EmbeddingsofCSV = getEmbeddingsofCSV(extractedData)

print("Total Rows = "+str(len(extractedData)))
# print("Total Cols = "+str(len(extractedData.columns.toList())))
print("len of text rep = "+str(len(textRep)))
print("30 or more yrs = "+str((extractedData['YearsCoding'] == '30 or more years').sum()))

while True:
  # print("Entered while")
  userInput = input("Enter inputQuestion and method of retrieval keyword or semantic separated by ; Enter Exit to quit. \n")

  if userInput == "Exit":
    break

  elif not userInput.__contains__(";"):
    print("\nError input. Input without ; wrong. ")

  else:
    if ';' in userInput:
      inputs = userInput.split(';')
      userQues = inputs[0].strip()
      methodofRetrieval = inputs[1].strip()
      # print("\n Question = "+str(userQues)+"\n retrieval method = "+str(methodofRetrieval))

      responseObtained = retriever(userQues, methodofRetrieval, textRep, extractedData, EmbeddingsofCSV)
      # print("\n Response Obtained = ")
      for res in responseObtained:
        print("\n Question = "+str(res['question']))
        print("\n Data Retrieval Method = "+str(res['dataretrievalMethod']))
        print("\n Result Obtained = "+str(res['response']))





[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


API loded
Total Rows = 1121
len of text rep = 1121
30 or more yrs = 81


KeyboardInterrupt: Interrupted by user

In [14]:
# Problem-1 MAX_ONE AI Generated Code

import random

# ====================================================================
# 1. PARAMETERS
# ====================================================================

LENGTH = 50           # Binary string length
POP_SIZE = 60         # Population size
GENERATIONS = 200    # Number of generations
CROSS_PROB = 0.6      # Crossover probability
MUT_PROB = 0.1        # Mutation probability per bit

# ====================================================================
# 2. HELPER FUNCTIONS (Initialisation, Fitness, Mutation, Crossover, Selection)
# ====================================================================

def generate_individual(length):
    """Generates a random binary string (list of 0s and 1s)."""
    return [random.choice([0, 1]) for _ in range(length)]

def initialize_population(pop_size, length):
    """Creates the initial random population."""
    return [generate_individual(length) for _ in range(pop_size)]

def evaluate_fitness(individual):
    """Calculates the fitness (number of '1's) of a single individual."""
    return sum(individual)

def evaluate_population_fitness(population):
    """Evaluates the fitness for all individuals in the population."""
    return [(individual, evaluate_fitness(individual)) for individual in population]

def selection(population_with_fitness, k=3):
    """
    Performs Tournament Selection (k=3) to choose one parent.
    The individual with the maximum fitness in the tournament wins.
    """
    # Randomly choose k individuals/fitness tuples
    tournament = random.choices(population_with_fitness, k=k)

    # The winner is the tuple with the highest fitness (index 1)
    winner = max(tournament, key=lambda item: item[1])

    # Return just the individual
    return winner[0]

def crossover(parent1, parent2, crossover_prob, length):
    """
    Performs one-point crossover with a given probability.
    Returns two offspring (children).
    """
    child1, child2 = parent1[:], parent2[:]  # Create copies

    if random.random() < crossover_prob:
        # Choose a random split point
        crossover_point = random.randint(1, length - 1)

        # Swap the segments
        child1 = parent1[:crossover_point] + parent2[crossover_point:]
        child2 = parent2[:crossover_point] + parent1[crossover_point:]

    return child1, child2

def mutation(individual, mutation_prob):
    """
    Performs bit-flip mutation on an individual with a given probability per bit.
    """
    mutated_individual = individual[:] # Create a copy

    for i in range(len(mutated_individual)):
        if random.random() < mutation_prob:
            # Flip the bit: 1 becomes 0, 0 becomes 1
            mutated_individual[i] = 1 - mutated_individual[i]

    return mutated_individual

# ====================================================================
# 3. MAIN EVOLUTIONARY ALGORITHM LOOP
# ====================================================================

def solve_max_one():
    """
    The main Evolutionary Algorithm loop for the MAX-ONE problem.
    """

    # INITIALISATION
    current_population = initialize_population(POP_SIZE, LENGTH)
    print(f"Algorithm Initialized: {LENGTH} bits, {POP_SIZE} individuals.")

    # Tracking variables
    best_individual = []
    max_fitness_overall = 0
    generation_found = 0

    print("-" * 50)

    for gen in range(1, GENERATIONS + 1):

        # FITNESS EVALUATION
        pop_with_fitness = evaluate_population_fitness(current_population)

        # Find the best fitness in the current generation
        current_max_result = max(pop_with_fitness, key=lambda item: item[1])
        current_max_fitness = current_max_result[1]
        current_best_individual = current_max_result[0]

        # Update the overall best solution
        if current_max_fitness > max_fitness_overall:
            max_fitness_overall = current_max_fitness
            best_individual = current_best_individual
            generation_found = gen

        # Log progress
        print(f"Gen {gen}/{GENERATIONS}: Max Fitness = {current_max_fitness}, Overall Best = {max_fitness_overall}")

        # Stopping Condition: Check if the optimum is found (all 1's)
        if max_fitness_overall == LENGTH:
            print("\n**Optimum Found!**")
            break

        # Create the new population for the next generation
        next_population = []

        # ELITISM: Copy the single best individual directly to the next generation
        next_population.append(current_best_individual[:])

        # Fill the rest of the next population
        while len(next_population) < POP_SIZE:

            # SELECTION: Choose two parents via tournament selection
            parent1 = selection(pop_with_fitness)
            parent2 = selection(pop_with_fitness)

            # CROSSOVER: Produce two children
            child1, child2 = crossover(parent1, parent2, CROSS_PROB, LENGTH)

            # MUTATION: Introduce random changes
            child1 = mutation(child1, MUT_PROB)
            child2 = mutation(child2, MUT_PROB)

            # Add children to the new population
            next_population.append(child1)
            # Ensure we don't exceed the defined population size
            if len(next_population) < POP_SIZE:
                 next_population.append(child2)

        # Move to the next generation
        current_population = next_population

    print("-" * 50)
    print(f"**EA Summary**")
    print(f"Finished after {gen} generations (Optimum reached in Gen {generation_found}).")
    print(f"Best Fitness Achieved: **{max_fitness_overall}/{LENGTH}**")
    print(f"Best Individual (String): **{''.join(map(str, best_individual))}**")

# --- Execute the main function ---
if __name__ == "__main__":
    solve_max_one()

Algorithm Initialized: 50 bits, 60 individuals.
--------------------------------------------------
Gen 1/200: Max Fitness = 33, Overall Best = 33
Gen 2/200: Max Fitness = 34, Overall Best = 34
Gen 3/200: Max Fitness = 36, Overall Best = 36
Gen 4/200: Max Fitness = 36, Overall Best = 36
Gen 5/200: Max Fitness = 39, Overall Best = 39
Gen 6/200: Max Fitness = 39, Overall Best = 39
Gen 7/200: Max Fitness = 39, Overall Best = 39
Gen 8/200: Max Fitness = 39, Overall Best = 39
Gen 9/200: Max Fitness = 39, Overall Best = 39
Gen 10/200: Max Fitness = 39, Overall Best = 39
Gen 11/200: Max Fitness = 39, Overall Best = 39
Gen 12/200: Max Fitness = 39, Overall Best = 39
Gen 13/200: Max Fitness = 41, Overall Best = 41
Gen 14/200: Max Fitness = 41, Overall Best = 41
Gen 15/200: Max Fitness = 41, Overall Best = 41
Gen 16/200: Max Fitness = 41, Overall Best = 41
Gen 17/200: Max Fitness = 41, Overall Best = 41
Gen 18/200: Max Fitness = 42, Overall Best = 42
Gen 19/200: Max Fitness = 42, Overall Best = 4

In [2]:
# First Problem - Given MAX_ONE code

import numpy as np

# Information
l = 50
n = 20
num_gens = 100
crossover_prob = 0.6

# Initialization
pop = np.random.randint(0, 2, (n, l))

print('Initial poputation:')
print(pop)

# Fitness
def fitness(X):
	return np.sum(X, axis = 1)

# Crossover
def crossover(parent1, parent2):
	n = len(parent1)
	i = np.random.randint(n)
	j = np.random.randint(n)
	if i > j:
		v = i
		i = j
		j = v
	child1 = np.zeros((n))
	child2 = np.zeros((n))
	child1[:i] = parent1[:i]
	child1[j+1:] = parent1[j+1:]
	child2[:i] = parent2[:i]
	child2[j+1:] = parent2[j+1:]
	child1[i:j+1] = parent2[i:j+1]
	child2[i:j+1] = parent1[i:j+1]
	return child1, child2

# Mutation
def mutate(x, mutation_prob = 0.1):
	for i in range(len(x)):
		prob = np.random.rand()
		if prob < mutation_prob:
			x[i] = (x[i] + 1) % 2
	return x

# Genetic Algorithm
for gen in range(num_gens):
	print('Generation', gen + 1, ':')
	print('- Maximum:', np.max(fitness(pop)))

	# Cross-over & Mutation
	offsprings = []
	for i in range(n):
		for j in range(i + 1, n):
			prob = np.random.rand()
			if prob < crossover_prob:
				child1, child2 = crossover(pop[i, :], pop[j, :])
				child1 = mutate(child1)
				child2 = mutate(child2)
				offsprings.append(child1)
				offsprings.append(child2)

	# Deterministic selection
	combine = []
	f = fitness(pop)
	for i in range(n):
		combine.append((- f[i], pop[i]))

	for i in range(len(offsprings)):
		f = fitness(np.reshape(offsprings[i], (1, l)))[0]
		combine.append((- f, offsprings[i]))

	combine.sort(key = lambda x: x[0])

	selected = []
	for i in range(n):
		selected.append(combine[i][1])
	pop = np.array(selected)



Initial poputation:
[[0 1 0 1 0 1 0 0 0 1 1 1 1 0 1 1 0 1 1 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 0 1
  0 1 0 0 0 0 1 0 1 1 1 0 1 1]
 [1 0 0 1 0 0 1 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 0 1 1 0 1 1 1 0 1 1 0 1 0
  1 0 0 0 0 1 1 0 0 0 1 0 1 1]
 [0 0 0 1 0 0 0 1 0 1 0 1 0 1 0 0 1 0 1 0 1 1 1 1 0 0 1 1 0 0 0 0 0 1 1 1
  1 0 0 0 0 1 1 0 1 1 1 1 1 1]
 [1 1 1 0 1 1 1 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 0 1 0 0 1 0 1 1 1 0 0 1 0
  1 0 0 0 1 1 1 0 1 1 0 0 0 0]
 [0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 1 1 1 0 1 0 0 0 0 0
  0 0 0 1 1 1 0 0 1 0 0 1 1 0]
 [1 0 0 1 1 0 0 0 1 0 1 1 1 0 0 1 1 1 0 1 1 0 0 1 1 1 0 0 1 1 0 0 0 1 0 1
  0 1 0 1 0 1 0 0 1 1 0 1 1 1]
 [0 0 1 1 1 0 1 1 0 0 0 1 0 0 1 1 1 0 0 0 1 0 1 0 0 1 0 0 1 1 0 1 1 1 1 0
  0 0 0 0 1 0 1 0 1 0 0 1 1 1]
 [1 1 0 0 0 1 0 0 0 1 1 0 0 1 1 0 1 0 1 1 0 1 1 1 1 1 1 0 1 0 1 1 0 1 1 1
  1 1 0 0 1 0 0 0 0 1 0 1 1 0]
 [0 0 1 1 0 0 0 1 0 0 1 0 0 0 1 1 1 1 1 0 0 0 0 1 1 0 1 1 1 0 1 1 0 0 1 0
  0 0 0 0 1 1 0 1 0 0 1 1 0 0]
 [0 0 0 0 1 1 1 0 1 0 0 1 1 1 1 0 1